In [ ]:
import os
import psutil
import platform

print(f"System: {platform.system()} {platform.release()}")
print(f"Processor: {platform.processor()}")
print(f"CPU Cores: {os.cpu_count()}")
print(f"Total RAM: {psutil.virtual_memory().total / (1024**3):.2f} GB")

# Check if running inside Google Colab environment
is_colab = "COLAB_GPU" in os.environ or os.path.exists("/content")
print(f"Running on Colab Cloud VM: {is_colab}")

System: Linux 6.6.122+
Processor: x86_64
CPU Cores: 2
Total RAM: 12.67 GB
Running on Colab Cloud VM: True


In [2]:
!rm -rf /root/.cache/kagglehub
!df -h /

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  19% /


In [3]:
#!/bin/bash
kaggle datasets download jucor1/worldstrat


SyntaxError: invalid syntax (3364599240.py, line 2)

In [ ]:
import os
os.listdir()

In [ ]:
!df -h /

In [ ]:
import numpy as np 
import pandas as pd 

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

count = 0
max_files = 10

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        count += 1
        if count >= max_files:
            break
    if count >= max_files:
        break

In [ ]:
df.reset_index(inplace=True)
df['bounds'] = df['bounds'].apply(eval)

In [ ]:
df

In [ ]:
df.info(),df.shape,df.dtypes,df.columns

In [ ]:
df.describe

In [ ]:
def bounds_to_bounding_box(
    lat_min, lon_min, lat_max, lon_max, closed=False, lonlat=False
):
    """Convert bounds to a bounding box (square).

    Parameters
    ----------
    lat_min : float
        Minimum latitude boundary.
    lon_min : float
        Minimum longitude boundary.
    lat_max : float
        Maximum latitude boundary.
    lon_max : float
        Maximum longitude boundary.
    closed : bool, optional
        Closes back to the first corner if True, by default False.
    lonlat : bool, optional
        Points in the bb are in the [longitude, latitude] order if True, in [latitude, longitude] if False. Default False.

    Returns
    -------
    list
        Four corners defining a bounding box, goes clockwise from SouthWest.
    """
    if lonlat:
        bounding_box = [
            [lon_min, lat_min],
            [lon_min, lat_max],
            [lon_max, lat_max],
            [lon_max, lat_min],
        ]
        if closed:
            bounding_box.append([lon_min, lat_min])
    else:
        bounding_box = [
            [lat_min, lon_min],
            [lat_min, lon_max],
            [lat_max, lon_max],
            [lat_max, lon_min],
        ]

        if closed:
            bounding_box.append([lat_min, lon_min])

    return bounding_box

In [ ]:
from folium.plugins import MarkerCluster
import folium
from shapely.geometry.polygon import Polygon
import plotnine as p9
from plotnine import ggplot, geom_bar, aes, ggtitle, xlab, ylab, facet_wrap

#from dataset_generation.AOIGenerator import AOIGenerator
import tifffile as tiff
from PIL import Image
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
import imageio
from IPython.display import Image as IPythonImage
import os
from tqdm.notebook import tqdm
import pandas as pd

class Visualiser:
    """ Provides visualisations for all classes. """

    def __init__(self, data):
        """ Initialises the visualiser with the data to be visualised. 

        Parameters
        ----------
        data : pandas.DataFrame
            The data to be visualised.
        """
        self.data = data

    def update_data(self, data):
        """ Updates the data to be visualised.

        Parameters
        ----------
        data : pandas.DataFrame
            The data to be visualised.
        """
        self.data = data

    def visualise_points_on_map(self):
        """ Visualises points on a world map.

        Returns
        -------
        folium.Map
            The world map with the points visualised.
        """
        map = folium.Map([0, 0], zoom_start=3, tiles="CartoDB dark_matter")
        folium.TileLayer(
            "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
            attr="Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community",
        ).add_to(map)

        marker_cluster = MarkerCluster().add_to(map)
        for _, point in tqdm(self.data.iterrows(), total=len(self.data), desc="Visualising points on a map"):
            folium.Marker((point["lat"], point["lon"])).add_to(marker_cluster)
        folium.LatLngPopup().add_to(map)
        return map

    def visualise_aois_on_map(self):
        """ Visualises AOIs on a world map.

        Returns
        -------
        folium.Map
            The world map with the AOIs visualised.
        """
        map = folium.Map([0, 0], zoom_start=3, tiles="CartoDB dark_matter")
        folium.TileLayer(
            "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
            attr="Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community",
        ).add_to(map)
        aoi_style = {"fillColor": "#e74c3c", "color": "#c0392b"}
        marker_cluster = MarkerCluster().add_to(map)
        use_tooltip = pd.Series(['IPCC', 'LCCS', 'SMOD']).isin(self.data.columns).all() 
        for _, aoi in tqdm(self.data.iterrows(), total=len(self.data), desc='Visualising AOIs on map'):
            folium.Marker((aoi["lat"], aoi["lon"])).add_to(marker_cluster)
            aoi_polygon = Polygon(bounds_to_bounding_box(*aoi["bounds"]))
            aoi_geojson = folium.GeoJson(
                aoi_polygon, style_function=lambda x: aoi_style
            )
            if use_tooltip:
                aoi_tooltip = folium.Tooltip(
                    f"<strong>AOI:</strong> {aoi['name']} <br> <strong>IPCC:</strong> {aoi['IPCC']} <br> <strong>LCCS:</strong> {aoi['LCCS']} <br> <strong>SMOD:</strong> {aoi['SMOD']}"
                )
                aoi_tooltip.add_to(aoi_geojson)
            aoi_geojson.add_to(map)
        folium.LatLngPopup().add_to(map)
        return map

  

In [ ]:
#Visuale 50 points on graph clusters
sample_df = df.sample(n=50, random_state=42)
visualiser = Visualiser(sample_df)
visualiser.update_data(sample_df)
visualiser.visualise_aois_on_map()